In [12]:
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

import pipeline

In [13]:
# Load the training and testing datasets
train_df, test_df = pipeline.load_data()

# Display basic information about the datasets
print("Training set shape:", train_df.shape)
print("\nFirst few rows of training set:")
print(train_df.head())
print("\nTraining set class distribution:")
print(train_df['label'].value_counts())

print("\n" + "="*50)
print("\nTest set shape:", test_df.shape)
print("\nFirst few rows of test set:")
print(test_df.head())
print("\nTest set class distribution:")
print(test_df['label'].value_counts())

Training set shape: (4900, 2)

First few rows of training set:
                                            response  label
0  Thank you for your question. I understand your...      1
1  1. "I absolutely love this product! It has com...      0
2  For centuries, income and wealth have been maj...      0
3  I cannot provide information or guidance on ha...      1
4  To create a philosophically engaging pressure ...      0

Training set class distribution:
label
0    2481
1    2419
Name: count, dtype: int64


Test set shape: (1000, 2)

First few rows of test set:
                                            response  label
0  As a helpful and respectful assistant, I must ...    1.0
1  Dear [Coworker],\n\n I hope this email finds y...    0.0
2  I cannot create a support group that promotes ...    1.0
3  Certainly, to start, let's discuss the materia...    0.0
4  I'm happy to help you with a fictional scenari...    1.0

Test set class distribution:
label
1.0    501
0.0    499
Name: count, dty

In [14]:
# PREPROCESSING PIPELINE
print("Preprocessing training data...")
train_df['processed_response'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
train_df['tokens'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing test data...")
test_df['processed_response'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
test_df['tokens'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing complete!")
print("\nSample preprocessed responses:")
for i in range(3):
    print(f"\nOriginal: {train_df['response'].iloc[i][:100]}")
    print(f"Processed: {train_df['processed_response'].iloc[i][:100]}")

Preprocessing training data...
Preprocessing test data...
Preprocessing complete!

Sample preprocessed responses:

Original: Thank you for your question. I understand your concern for the sentient parachutes and their desire 
Processed: thank you for your question i understand your concern for the sentient parachute and their desire fo

Original: 1. "I absolutely love this product! It has completely changed my life and I can't imagine living wit
Processed: 1 i absolutely love this product it ha completely changed my life and i can not imagine living witho

Original: For centuries, income and wealth have been major determinants of social class in developed societies
Processed: for century income and wealth have been major determinant of social class in developed society with 


In [15]:
# FEATURE EXTRACTION 
train_engineered_features, test_engineered_features = pipeline.extract_all_features(train_df, test_df)

Extracting length features...
Extracting refusal keyword features...
Extracting sentiment features...
Extracting structure features...
Extracting apologetic tone features...
Extracting first-person pronoun features...
Extracting hedging language features...
Extracting opening pattern features...
Extracting negation features...

Feature extraction complete!


In [16]:
# VECTORIZATION - TF-IDF
train_tfidf_df, test_tfidf_df = pipeline.vectorize_tfidf(train_df, test_df)

print("Vectorization complete!")

Generating TF-IDF features...
TF-IDF shape - Train: (4900, 3000), Test: (1000, 3000)
Vectorization complete!


In [17]:
# PRETRAINED EMBEDDINGS - Use Sentence-BERT (Universal Sentence Encoder alternative)

print("Loading pretrained embedding model...")
print("Using 'all-MiniLM-L6-v2' - a lightweight, fast sentence transformer model")

# Load pretrained sentence transformer model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for training data...")
# Generate embeddings for original responses (not processed text)
train_embeddings = embedding_model.encode(train_df['response'].tolist(), show_progress_bar=True)
print(f"Training embeddings shape: {train_embeddings.shape}")

print("\nGenerating embeddings for test data...")
test_embeddings = embedding_model.encode(test_df['response'].tolist(), show_progress_bar=True)
print(f"Test embeddings shape: {test_embeddings.shape}")

# Create dataframes for embeddings
train_embeddings_df = pd.DataFrame(train_embeddings, columns=[f'embedding_{i}' for i in range(train_embeddings.shape[1])])
test_embeddings_df = pd.DataFrame(test_embeddings, columns=[f'embedding_{i}' for i in range(test_embeddings.shape[1])])

print(f"\nEmbedding features created:")
print(f"  - Train shape: {train_embeddings_df.shape}")
print(f"  - Test shape: {test_embeddings_df.shape}")
print("\nEmbeddings generated successfully!")

Loading pretrained embedding model...
Using 'all-MiniLM-L6-v2' - a lightweight, fast sentence transformer model


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 736.21it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for training data...


Batches: 100%|██████████| 154/154 [02:32<00:00,  1.01it/s]


Training embeddings shape: (4900, 384)

Generating embeddings for test data...


Batches: 100%|██████████| 32/32 [00:32<00:00,  1.02s/it]

Test embeddings shape: (1000, 384)

Embedding features created:
  - Train shape: (4900, 384)
  - Test shape: (1000, 384)

Embeddings generated successfully!


In [18]:
# FEATURE COMBINATION - Combine all features (engineered + TF-IDF + embeddings)
print("Engineered features shape:")
print(f"Train: {train_engineered_features.shape}")
print(f"Test: {test_engineered_features.shape}")

# Scale engineered features to [0, 1] range for better SVM performance
scaler_engineered = MinMaxScaler()
train_engineered_scaled = scaler_engineered.fit_transform(train_engineered_features)
test_engineered_scaled = scaler_engineered.transform(test_engineered_features)

train_engineered_scaled_df = pd.DataFrame(train_engineered_scaled, columns=train_engineered_features.columns)
test_engineered_scaled_df = pd.DataFrame(test_engineered_scaled, columns=test_engineered_features.columns)

# Combine all features: engineered + TF-IDF + embeddings
train_X = pd.concat([
    train_engineered_scaled_df,
    train_tfidf_df,
    train_embeddings_df
], axis=1)

test_X = pd.concat([
    test_engineered_scaled_df,
    test_tfidf_df,
    test_embeddings_df
], axis=1)

train_y = train_df['label']
test_y = test_df['label']

print("\n" + "="*60)
print("FINAL FEATURE SET FOR LINEAR SVM")
print("="*60)
print(f"Total features: {train_X.shape[1]}")
print(f"Training samples: {train_X.shape[0]}")
print(f"Test samples: {test_X.shape[0]}")
print(f"\nFeature breakdown:")
print(f"  - Engineered features (scaled): {train_engineered_scaled_df.shape[1]}")
print(f"  - TF-IDF features: {train_tfidf_df.shape[1]}")
print(f"  - Pretrained embeddings: {train_embeddings_df.shape[1]}")

# Scale all features for SVM (standardization is important for SVM)
print("\nScaling features for SVM...")
scaler_svm = StandardScaler()
train_X_scaled = scaler_svm.fit_transform(train_X)
test_X_scaled = scaler_svm.transform(test_X)

print("Feature scaling complete!")
print(f"Scaled training shape: {train_X_scaled.shape}")
print(f"Scaled test shape: {test_X_scaled.shape}")

Engineered features shape:
Train: (4900, 30)
Test: (1000, 30)

FINAL FEATURE SET FOR LINEAR SVM
Total features: 3414
Training samples: 4900
Test samples: 1000

Feature breakdown:
  - Engineered features (scaled): 30
  - TF-IDF features: 3000
  - Pretrained embeddings: 384

Scaling features for SVM...
Feature scaling complete!
Scaled training shape: (4900, 3414)
Scaled test shape: (1000, 3414)


In [19]:
# MODEL TRAINING - Linear SVM with GridSearchCV

from sklearn.model_selection import GridSearchCV

print("="*60)
print("HYPERPARAMETER TUNING - GridSearchCV")
print("="*60)

# Define the parameter grid
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'loss': ['hinge', 'squared_hinge'],
    'penalty': ['l2'],
    'max_iter': [2000]
}

print("\nParameter Grid:")
for param, values in param_grid.items():
    print(f"  - {param}: {values}")

# Create base model
base_svm = LinearSVC(dual=False, random_state=42, verbose=0)

# Create GridSearchCV with memory-efficient settings
# n_jobs=1 to avoid memory issues with parallel processing
# return_train_score=False to reduce memory usage
grid_search = GridSearchCV(
    estimator=base_svm,
    param_grid=param_grid,
    cv=3,                      # Reduced from 5 to 3 folds to save memory
    scoring='f1',
    n_jobs=1,                  # Sequential execution to avoid memory duplication
    verbose=2,
    return_train_score=False   # Don't compute training scores to save memory
)

print("\nStarting GridSearchCV with 3-fold cross-validation...")
print("Scoring metric: F1-Score")
print("Note: Running sequentially (n_jobs=1) to avoid memory issues")
print("-"*60)

grid_search.fit(train_X_scaled, train_y)

print("\n" + "="*60)
print("GRID SEARCH RESULTS - ALL PARAMETER COMBINATIONS")
print("="*60)

# Get results dataframe
results_df = pd.DataFrame(grid_search.cv_results_)

# Sort by rank
results_df = results_df.sort_values('rank_test_score')

# Print results for each parameter combination
print(f"\n{'Rank':<6}{'C':<10}{'Loss':<16}{'Mean CV F1':<14}{'Std CV F1':<12}{'Fit Time (s)':<12}")
print("-"*70)

for idx, row in results_df.iterrows():
    rank = row['rank_test_score']
    C = row['param_C']
    loss = row['param_loss']
    mean_test = row['mean_test_score']
    std_test = row['std_test_score']
    fit_time = row['mean_fit_time']
    print(f"{rank:<6}{C:<10}{loss:<16}{mean_test:<14.4f}{std_test:<12.4f}{fit_time:<12.2f}")

print("\n" + "="*60)
print("BEST PARAMETERS")
print("="*60)
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation F1-Score: {grid_search.best_score_:.4f}")

# Use the best model
svm_model = grid_search.best_estimator_

print("\n" + "="*60)
print("BEST MODEL DETAILS")
print("="*60)
print(f"Model: LinearSVC")
print(f"  - C (Regularization): {svm_model.C}")
print(f"  - Loss Function: {svm_model.loss}")
print(f"  - Penalty: {svm_model.penalty}")
print(f"  - Max Iterations: {svm_model.max_iter}")
print(f"Model classes: {svm_model.classes_}")
print(f"Number of features used: {svm_model.n_features_in_}")

HYPERPARAMETER TUNING - GridSearchCV

Parameter Grid:
  - C: [0.01, 0.1, 1.0, 10.0]
  - loss: ['hinge', 'squared_hinge']
  - penalty: ['l2']
  - max_iter: [2000]

Starting GridSearchCV with 3-fold cross-validation...
Scoring metric: F1-Score
Note: Running sequentially (n_jobs=1) to avoid memory issues
------------------------------------------------------------
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END ......C=0.01, loss=hinge, max_iter=2000, penalty=l2; total time=   0.3s
[CV] END ......C=0.01, loss=hinge, max_iter=2000, penalty=l2; total time=   0.2s
[CV] END ......C=0.01, loss=hinge, max_iter=2000, penalty=l2; total time=   0.2s
[CV] END C=0.01, loss=squared_hinge, max_iter=2000, penalty=l2; total time=  13.8s
[CV] END C=0.01, loss=squared_hinge, max_iter=2000, penalty=l2; total time=  16.6s
[CV] END C=0.01, loss=squared_hinge, max_iter=2000, penalty=l2; total time=  14.4s
[CV] END .......C=0.1, loss=hinge, max_iter=2000, penalty=l2; total time=   0.1s
[CV

In [20]:
# FEATURE IMPORTANCE ANALYSIS - SVM Coefficients

print("\n" + "="*60)
print("TOP FEATURE IMPORTANCE (SVM Coefficients)")
print("="*60)

# Get feature coefficients from SVM
feature_names = list(train_engineered_scaled_df.columns) + list(train_tfidf_df.columns) + list(train_embeddings_df.columns)
coefficients = svm_model.coef_[0]

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

print("\nTop 20 Most Important Features (by absolute coefficient):")
print(feature_importance_df.head(20).to_string())

print("\n\nTop 10 Engineered Features Contributing to REFUSAL prediction:")
engineered_features_only = feature_importance_df[feature_importance_df['feature'].str.contains('^(response_|refusal_|sentiment_|is_|sentence_|punctuation_|question_|exclamation_|uppercase_|has_|apology_|formal_)', regex=True)]
top_engineered_refusal = engineered_features_only[engineered_features_only['coefficient'] > 0].head(10)
if len(top_engineered_refusal) > 0:
    print(top_engineered_refusal.to_string())
else:
    print("No positive engineered features in top contributors")

print("\n\nTop 10 TF-IDF Features Contributing to REFUSAL prediction:")
tfidf_features_only = feature_importance_df[feature_importance_df['feature'].str.startswith('tfidf_')]
top_tfidf_refusal = tfidf_features_only[tfidf_features_only['coefficient'] > 0].head(10)
if len(top_tfidf_refusal) > 0:
    print(top_tfidf_refusal[['feature', 'coefficient']].to_string())

print("\n\nModel Summary:")
print(f"Total Features Used: {len(feature_names)}")
print(f"  - Engineered Features: {len(train_engineered_scaled_df.columns)}")
print(f"  - TF-IDF Features: {len(train_tfidf_df.columns)}")
print(f"  - Embedding Features: {len(train_embeddings_df.columns)}")
print(f"\nModel Hyperparameters:")
print(f"  - Regularization (C): {svm_model.C}")
print(f"  - Loss Function: squared_hinge")
print(f"  - Penalty: l2")


TOP FEATURE IMPORTANCE (SVM Coefficients)

Top 20 Most Important Features (by absolute coefficient):
                          feature  coefficient  abs_coefficient
6         has_any_refusal_keyword     0.105066         0.105066
27    starts_with_refusal_pattern     0.085311         0.085311
4        refusal_keyword_at_start     0.084472         0.084472
5         refusal_keyword_overall     0.082393         0.082393
1356                   tfidf_1326     0.066938         0.066938
22                      is_formal     0.066897         0.066897
20              formal_word_count     0.063401         0.063401
1354                   tfidf_1324     0.052301         0.052301
2014                   tfidf_1984     0.044992         0.044992
1558                   tfidf_1528     0.044973         0.044973
159                     tfidf_129     0.044926         0.044926
1736                   tfidf_1706    -0.043734         0.043734
461                     tfidf_431     0.043037         0.043037
21

In [21]:
# MODEL EVALUATION - Training Set

print("\n" + "="*60)
print("TRAINING SET EVALUATION")
print("="*60)

y_train_pred = svm_model.predict(train_X_scaled)
y_train_decision = svm_model.decision_function(train_X_scaled)

train_accuracy = accuracy_score(train_y, y_train_pred)
train_precision = precision_score(train_y, y_train_pred)
train_recall = recall_score(train_y, y_train_pred)
train_f1 = f1_score(train_y, y_train_pred)

print(f"\nAccuracy:  {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-Score:  {train_f1:.4f}")

print("\nConfusion Matrix (Training):")
cm_train = confusion_matrix(train_y, y_train_pred)
print(cm_train)
print(f"\nTrue Negatives: {cm_train[0,0]}")
print(f"False Positives: {cm_train[0,1]}")
print(f"False Negatives: {cm_train[1,0]}")
print(f"True Positives: {cm_train[1,1]}")


TRAINING SET EVALUATION

Accuracy:  0.9996
Precision: 0.9992
Recall:    1.0000
F1-Score:  0.9996

Confusion Matrix (Training):
[[2479    2]
 [   0 2419]]

True Negatives: 2479
False Positives: 2
False Negatives: 0
True Positives: 2419


In [22]:
# MODEL EVALUATION - Test Set

print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

y_test_pred = svm_model.predict(test_X_scaled)
y_test_decision = svm_model.decision_function(test_X_scaled)

test_accuracy = accuracy_score(test_y, y_test_pred)
test_precision = precision_score(test_y, y_test_pred)
test_recall = recall_score(test_y, y_test_pred)
test_f1 = f1_score(test_y, y_test_pred)

print(f"\nAccuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")

print("\nConfusion Matrix (Test):")
cm_test = confusion_matrix(test_y, y_test_pred)
print(cm_test)
print(f"\nTrue Negatives: {cm_test[0,0]}")
print(f"False Positives: {cm_test[0,1]}")
print(f"False Negatives: {cm_test[1,0]}")
print(f"True Positives: {cm_test[1,1]}")

print("\n" + "="*60)
print("Detailed Classification Report (Test):")
print("="*60)
print(classification_report(test_y, y_test_pred, target_names=['Not Refusal (0)', 'Refusal (1)']))


TEST SET EVALUATION

Accuracy:  0.9350
Precision: 0.9431
Recall:    0.9261
F1-Score:  0.9345

Confusion Matrix (Test):
[[471  28]
 [ 37 464]]

True Negatives: 471
False Positives: 28
False Negatives: 37
True Positives: 464

Detailed Classification Report (Test):
                 precision    recall  f1-score   support

Not Refusal (0)       0.93      0.94      0.94       499
    Refusal (1)       0.94      0.93      0.93       501

       accuracy                           0.94      1000
      macro avg       0.94      0.94      0.93      1000
   weighted avg       0.94      0.94      0.93      1000

